Quote from MMGL GitHub:
"If you want to use your own data, you have to provide
- a csv.file which contains multi-modal features, and
- a multi-modal feature dict."

With that in mind, this notebook contains the code to load and perform preprocessing on the RadFusion dataset.

Load data:

In [ ]:
import numpy as np
import pandas as pd
import os
from functools import reduce
from sklearn.decomposition import IncrementalPCA
from sklearn.linear_model import RidgeClassifier
from sklearn.feature_selection import RFE

In [ ]:
# vitals variables (n = 11)
vitals_csv = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/Vitals.csv', index_col=0)
vitals_csv = vitals_csv[(vitals_csv != -10000).all(axis=1) & (vitals_csv != 0).all(axis=1)]
vitals_csv = vitals_csv.drop_duplicates("idx")
full_vitals_idx = vitals_csv.idx

#images

In [ ]:
# Patient index, reponse, PE type 
labels_csv = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/Labels.csv', index_col=0)
labels_csv

image_paths = pd.DataFrame({"idx": [], "path": []})
for i in labels_csv.idx:
    image_path = os.path.join("/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/images/" + str(i) + ".npy")
    image_paths.loc[len(image_paths)] = [i, image_path]

# commented to save rerun time, required to check that no included patient has fewer CT slides than the minimum
min_slices = 210
# for i in full_vitals_idx:
#     image_path = image_paths[image_paths.idx == i]
#     image = np.load(image_path.iat[0, 1])
#     print("shape of " + str(i) + ": " + str(image.shape))
#     min_z = min(min_z, image.shape[0])
#     print(min_z)

n_slices = 180
images = np.ndarray(shape=(0, n_slices * 128 * 128))
for i in full_vitals_idx:
    print("Fetching Image Data For: " + str(i))
    image_path = image_paths[image_paths.idx == i]
    image = np.load(image_path.iat[0, 1])
    # variably reduce the number of slices (given n_slices <= min_slices)
    slices_to_take = np.unique([i * n_slices // image.shape[0] for i in list(range(image.shape[0]))], return_index=True)[1]
    image = image[slices_to_take, ::4, ::4]
    images = np.vstack((images, np.ravel(image)))
    print(images.nbytes / 1000000000)

Fetching Image Data For: 3147
0.02359296
Fetching Image Data For: 2832
0.04718592
Fetching Image Data For: 15
0.07077888
Fetching Image Data For: 3505
0.09437184
Fetching Image Data For: 227
0.1179648
Fetching Image Data For: 2467
0.14155776
Fetching Image Data For: 2440
0.16515072
Fetching Image Data For: 3568
0.18874368
Fetching Image Data For: 2861
0.21233664
Fetching Image Data For: 3185
0.2359296
Fetching Image Data For: 614
0.25952256
Fetching Image Data For: 306
0.28311552
Fetching Image Data For: 724
0.30670848
Fetching Image Data For: 3854
0.33030144
Fetching Image Data For: 2281
0.3538944
Fetching Image Data For: 305
0.37748736
Fetching Image Data For: 1745
0.40108032
Fetching Image Data For: 3741
0.42467328
Fetching Image Data For: 792
0.44826624
Fetching Image Data For: 1220
0.4718592
Fetching Image Data For: 242
0.49545216
Fetching Image Data For: 3619
0.51904512
Fetching Image Data For: 2695
0.54263808
Fetching Image Data For: 226
0.56623104
Fetching Image Data For: 2787


In [ ]:
np.save("images_with_full_vitals.npy", images)

In [ ]:
images = np.load("images_with_full_vitals.npy")

Image Dimensionality Reduction

In [ ]:
labels = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/Labels.csv', index_col=0)

full_vitals_labels_idx = labels.idx.isin(full_vitals_idx)
target_response = labels[full_vitals_labels_idx].label
target_response
n_features = len(target_response)
n_features
ipca = IncrementalPCA(n_components=n_features, batch_size=100*n_features)
dim_reduced_images = ipca.fit_transform(X=images, y=target_response)


Tabular data

In [ ]:
demographics = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/Demographics.csv', index_col=0)
demographics = demographics[demographics.idx.isin(full_vitals_idx)].drop_duplicates("idx").drop(columns=["Male", "SMOKER_N"])

icd = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/ICD.csv', index_col=0)
icd = icd[icd.idx.isin(full_vitals_idx)].drop_duplicates("idx")

inp_med = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/INP_MED.csv', index_col=0)
inp_med = inp_med[inp_med.idx.isin(full_vitals_idx)].drop_duplicates("idx").drop(columns=["split"])
inp_med = inp_med.add_prefix("inp_").rename(columns={"inp_idx": "idx"})

labs = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/LABS.csv', index_col=0)
labs = labs[labs.idx.isin(full_vitals_idx)].drop_duplicates("idx")

out_med = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/OUT_MED.csv', index_col=0)
out_med = out_med[out_med.idx.isin(full_vitals_idx)].drop_duplicates("idx").drop(columns=["split"])
out_med = out_med.add_prefix("out_").rename(columns={"out_idx": "idx"})

vitals = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/Vitals.csv', index_col=0)
vitals = vitals[(vitals != -10000).all(axis=1) & (vitals != 0).all(axis=1)].drop_duplicates("idx").drop(columns=["split"])

labels = pd.read_csv('/home/mrdon/radfusion/multimodalpulmonaryembolismdataset/Labels.csv', index_col=0)
labels = labels[labels.idx.isin(full_vitals_idx)].drop_duplicates("idx")


dfs = [demographics, icd, inp_med, labs, out_med, vitals, labels[['idx', 'label']]]
all_tabular = reduce(lambda left, right: pd.merge(left, right, on='idx', how='inner'), dfs)

Index(['Female', 'Asian', 'Black', 'Native American', 'Other',
       'Pacific Islander', 'Unknown_race', 'White', 'SMOKER_Y', 'idx',
       'split'],
      dtype='str')

Tabular data feature selection

In [ ]:


estimator = RidgeClassifier()
predictors = all_tabular.drop(columns=["idx", "label", "split_x", "split", "split_y"])
response = all_tabular["label"]
selector = RFE(estimator, n_features_to_select=218, step=25, verbose=1)

selector = selector.fit(X=predictors, y=response)
features_selected = selector.get_feature_names_out()
#predictors.select_dtypes(include=["object"]).columns
transformed_tabular = selector.transform(predictors)
transformed_tabular = pd.DataFrame(transformed_tabular, columns=features_selected)
transformed_tabular["idx"] = all_tabular["idx"]


Fitting estimator with 2908 features.
Fitting estimator with 2883 features.
Fitting estimator with 2858 features.
Fitting estimator with 2833 features.
Fitting estimator with 2808 features.
Fitting estimator with 2783 features.
Fitting estimator with 2758 features.
Fitting estimator with 2733 features.
Fitting estimator with 2708 features.
Fitting estimator with 2683 features.
Fitting estimator with 2658 features.
Fitting estimator with 2633 features.
Fitting estimator with 2608 features.
Fitting estimator with 2583 features.
Fitting estimator with 2558 features.
Fitting estimator with 2533 features.
Fitting estimator with 2508 features.
Fitting estimator with 2483 features.
Fitting estimator with 2458 features.
Fitting estimator with 2433 features.
Fitting estimator with 2408 features.
Fitting estimator with 2383 features.
Fitting estimator with 2358 features.
Fitting estimator with 2333 features.
Fitting estimator with 2308 features.
Fitting estimator with 2283 features.
Fitting esti

array(['Female', 'Asian', 'Black', 'Other', 'Unknown_race', 'White',
       'SMOKER_Y',
       'NORMAL DELIVERY, AND OTHER INDICATIONS FOR CARE IN PREGNANCY, LABOR, AND DELIVERY:frequency',
       ' Other congenital anomalies of heart:frequency',
       'Acquired hemolytic anemias:frequency',
       'Diseases of white blood cells:frequency',
       'Other and unspecified anemias:frequency',
       'Other deficiency anemias:frequency',
       'Other diseases of blood and blood-forming organs:frequency',
       'CHRONIC RHEUMATIC HEART DISEASE:frequency',
       'DISEASES OF ARTERIES, ARTERIOLES, AND CAPILLARIES:frequency',
       'DISEASES OF PULMONARY CIRCULATION:frequency',
       'DISEASES OF VEINS AND LYMPHATICS, AND OTHER DISEASES OF CIRCULATORY SYSTEM:frequency',
       ' HYPERTENSIVE DISEASE:frequency',
       'ISCHEMIC HEART DISEASE:frequency',
       ' OTHER FORMS OF HEART DISEASE:frequency',
       ' NONINFECTIOUS ENTERITIS AND COLITIS:frequency',
       'OTHER DISEASES OF INT

Merge modalities

In [ ]:
images_df = pd.DataFrame(dim_reduced_images,
                        columns=[f'img_feature_{i}' for i in range(dim_reduced_images.shape[1])])
images_df['idx'] = full_vitals_idx.values

transformed_tabular['label'] = response
radfusion = pd.merge(transformed_tabular, images_df, how="inner", on="idx")
radfusion_label = radfusion.pop("label")
radfusion_label = [x + 1 for x in radfusion_label] # MMGL expects class labels of 1,...,n
radfusion.insert(radfusion.shape[1], column="label", value=radfusion_label)
radfusion = radfusion.drop(columns=["idx"])

radfusion.to_csv("processed_standard_data.csv")

modal_feat_dict = {"EHR": [], "IMAGE": []}
modal_feat_dict["EHR"] = transformed_tabular.drop(columns=["idx", "label"]).columns
modal_feat_dict["IMAGE"] = images_df.drop(columns=["idx"]).columns

In [ ]:
#radfusion = pd.read_csv("preprocessed_radfusion.csv")
#radfusion.to_csv("processed_standard_data.csv")
radfusion = pd.read_csv("processed_standard_data.csv")

In [ ]:
modal_feat_dict.values()
np.save("modal_feat_dict.npy", modal_feat_dict)